Assignment #6
- Alia Kasem 
- All Foundation of Empirical Research

Python section

In [0]:
# Checking the tables from the schema using Spark catalog. 
spark.catalog.listTables()

In [0]:
# Checking the tables from the schema 
flights = spark.table("flights")
display(flights)

In [0]:
flights = spark.table("flights") 

In [0]:
print(flights.columns)

In [0]:
# Percentage of missing values
from pyspark.sql.functions import col, count, when

total_rows = flights.count()

missing_stats = flights.select([
    (count(when(col(c).isNull(), c)) / total_rows * 100).alias(c)
    for c in ["dep_delay", "arr_delay", "air_time"]
])

display(missing_stats)

In [0]:

# Mean vs Median (arr_delay)
from pyspark.sql.functions import mean

# Mean
flights.select(mean("arr_delay").alias("mean_arr_delay")).show()

# Median (approx)
flights.selectExpr("percentile_approx(arr_delay, 0.5) AS median_arr_delay").show()

- The median (-5) is a better fill value than the mean (6.9) for arr_delay.
- The large difference between the mean and median indicates the data is heavily
- skewed by extreme positive delays (outliers). These outliers pull the mean upward,
- making it unrepresentative of a typical flight. The median, which is resistant
to outliers, better reflects the typical arrival delay.

`The negative median suggests that more than half of flights actually arrive early,
even though the average delay is positive due to a smaller number of very late flights.`

`Full Cleaning Pipeline`

In [0]:
# Remove duplicates
df = flights.dropDuplicates(["year", "month", "day", "carrier", "flight"])

In [0]:
# fillin missing dep_delay values with median 
median_dep = df.selectExpr("percentile_approx(dep_delay, 0.5)").collect()[0][0]

df = df.fillna({"dep_delay": median_dep})

In [0]:
# Flagging arr_delay outliers using IQR 
quantiles = df.approxQuantile("arr_delay", [0.25, 0.75], 0.01)
Q1, Q3 = quantiles
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

from pyspark.sql.functions import when

df = df.withColumn(
    "arr_delay_outlier",
    when((col("arr_delay") < lower) | (col("arr_delay") > upper), True).otherwise(False)
)

display(df)

In [0]:
# Vaildation check
results = {}

#  No duplicates remain
results["no_duplicates"] = df.count() == df.dropDuplicates(
    ["year", "month", "day", "carrier", "flight"]
).count()

#  No missing dep_delay
results["dep_delay_no_nulls"] = df.filter(col("dep_delay").isNull()).count() == 0

#  Outlier column exists and is boolean-like
results["outlier_flag_exists"] = "arr_delay_outlier" in df.columns

# Print results
for k, v in results.items():
    print(f"{k}: {'PASS' if v else 'FAIL'}")

In [0]:
from pyspark.sql import Row

validation_df = spark.createDataFrame([
    Row(check=k, result="PASS" if v else "FAIL")
    for k, v in results.items()
])

display(validation_df)

sql 

In [0]:
%sql
WITH deduped AS (
  -- (a) Proper deduplication using a stronger key
  -- flights are NOT uniquely identified without origin/dest
  SELECT *
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
             PARTITION BY year, month, day, carrier, flight, origin, dest
             ORDER BY sched_dep_time
           ) AS rn
    FROM flights
  ) t
  WHERE rn = 1
),

dep_median AS (
  -- (b) compute median for dep_delay
  SELECT PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY dep_delay) AS median_dep
  FROM deduped
),

filled AS (
  -- fill missing dep_delay with median
  SELECT
    d.*,
    COALESCE(d.dep_delay, m.median_dep) AS dep_delay_filled
  FROM deduped d
  CROSS JOIN dep_median m
),

iqr_stats AS (
  -- compute IQR bounds for arr_delay
  SELECT
    PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY arr_delay) AS Q1,
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY arr_delay) AS Q3
  FROM filled
),

final AS (
  -- flag outliers
  SELECT
    f.*,
    CASE
      WHEN f.arr_delay IS NULL THEN NULL
      WHEN f.arr_delay < (s.Q1 - 1.5 * (s.Q3 - s.Q1))
        OR f.arr_delay > (s.Q3 + 1.5 * (s.Q3 - s.Q1))
      THEN 1 ELSE 0
    END AS arr_delay_outlier
  FROM filled f
  CROSS JOIN iqr_stats s
)

-- =========================
-- VALIDATION REPORT
-- =========================
SELECT
  -- Check 1: no duplicates on improved key
  COUNT(*) = COUNT(DISTINCT year, month, day, carrier, flight, origin, dest)
    AS no_duplicates,

  -- Check 2: no missing dep_delay after fill
  SUM(CASE WHEN dep_delay_filled IS NULL THEN 1 ELSE 0 END) = 0
    AS no_null_dep_delay,

  -- Check 3: dataset is not empty
  COUNT(*) > 0
    AS dataset_not_empty,

  -- Extra sanity check: outlier flag exists
  MAX(arr_delay_outlier) IS NOT NULL
    AS outlier_flag_exists

FROM final;